1.Assignment Tasks

In [1]:
import pandas as pd
import numpy as np
import re

A. Load Data

In [2]:
data = pd.read_excel('dataset1.xls')

df = pd.DataFrame(data)
df.head()

,ticket_id,ticket_text,issue_type,urgency_level,product
0,1,Payment issue for my SmartWatch V2. I was unde...,Billing Problem,Medium,SmartWatch V2
1,2,Can you tell me more about the UltraClean Vacu...,General Inquiry,NaN,UltraClean Vacuum
2,3,I ordered SoundWave 300 but got EcoBreeze AC i...,Wrong Item,Medium,SoundWave 300
3,4,Facing installation issue with PhotoSnap Cam. ...,Installation Issue,Low,PhotoSnap Cam
4,5,Order #30903 for Vision LED TV is 13 days late...,Late Delivery,NaN,Vision LED TV


B. Handling Missing Data

In [3]:
# We drop rows where the primary feature 'ticket_text' is missing
df_clean = df.dropna().copy()
df_clean.isnull().sum()

ticket_id        0
ticket_text      0
issue_type       0
urgency_level    0
product          0
dtype: int64

C. Preprocessing Functions

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [5]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ramku\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ramku\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ramku\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ramku\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
# Initialize Lemmatizer and Stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [7]:
def clean_and_preprocess(text):
    # 1. Text Normalization: Convert to lowercase
    text = str(text).lower()
    
    # 2. Remove special characters and numbers
    # Keeping only alphabets and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 3. Tokenization
    tokens = text.split()
    
    # 4. Stopword Removal and Lemmatization
    processed_tokens = [
        lemmatizer.lemmatize(word) 
        for word in tokens 
        if word not in stop_words and len(word) > 2  # Removing very short words
    ]
    
    return " ".join(processed_tokens)



In [8]:
# Apply the preprocessing
df_clean['processed_text'] = df_clean['ticket_text'].apply(clean_and_preprocess)


D. Output Verification

In [9]:
print("Original Rows:", len(df))
print("Cleaned Rows:", len(df_clean))
print("\nSample Data:")
print(df_clean[['ticket_text', 'processed_text']].head())

Original Rows: 1000
Cleaned Rows: 826

Sample Data:
                                         ticket_text  \
0  Payment issue for my SmartWatch V2. I was unde...   
2  I ordered SoundWave 300 but got EcoBreeze AC i...   
3  Facing installation issue with PhotoSnap Cam. ...   
5  Can you tell me more about the PhotoSnap Cam w...   
6   is malfunction. It stopped working after just...   

                                      processed_text  
0         payment issue smartwatch underbilled order  
2  ordered soundwave got ecobreeze instead order ...  
3  facing installation issue photosnap cam setup ...  
5     tell photosnap cam warranty also available red  
6                    malfunction stopped working day  


2. Feature Engineering

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer
from scipy.sparse import hstack
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ramku\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

A. Text Vectorization (TF-IDF)

In [11]:
# We limit max_features to keep the dataset manageable (e.g., top 1000 words)
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

# Fit and transform the processed text
tfidf_matrix = tfidf_vectorizer.fit_transform(df_clean['processed_text'])

# Convert to DataFrame for visualization (optional, usually kept as sparse matrix)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print(f"TF-IDF Matrix Shape: {tfidf_df.shape}")

TF-IDF Matrix Shape: (826, 101)


B. Extract Additional Features

In [12]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    # Returns a compound score between -1 (Negative) and +1 (Positive)
    return sia.polarity_scores(str(text))['compound']

# Feature 1: Ticket Length (Character count of original text)
# Rationale: Technical logs or angry rants might be significantly longer.
df_clean['ticket_length'] = df_clean['ticket_text'].apply(len)

# Feature 2: Sentiment Score
# Rationale: 'High' urgency tickets often contain negative sentiment words (angry, fail, bad).
df_clean['sentiment_score'] = df_clean['ticket_text'].apply(get_sentiment)

C. Combine Features

In [13]:
# Combine metadata features with TF-IDF vectors
# Note: For many ML models, you can stack these horizontally.
meta_features = df_clean[['ticket_length', 'sentiment_score']].values

print("\n--- Feature Engineering Results ---")
print(df_clean[['ticket_text', 'ticket_length', 'sentiment_score']].head())


--- Feature Engineering Results ---
                                         ticket_text  ticket_length  \
0  Payment issue for my SmartWatch V2. I was unde...             71   
2  I ordered SoundWave 300 but got EcoBreeze AC i...             80   
3  Facing installation issue with PhotoSnap Cam. ...             68   
5  Can you tell me more about the PhotoSnap Cam w...             84   
6   is malfunction. It stopped working after just...             54   

   sentiment_score  
0           0.0000  
2           0.1154  
3          -0.4215  
5           0.0000  
6          -0.2263  


In [14]:
# Combine with TF-IDF Matrix
# hstack (horizontal stack) appends the metadata columns to the right side of the TF-IDF matrix
X_combined = hstack([tfidf_matrix, meta_features])

# Verification
print(f"Shape of TF-IDF Matrix: {tfidf_matrix.shape}")
print(f"Shape of Metadata:      {meta_features.shape}")
print(f"Shape of Combined Data: {X_combined.shape}")

Shape of TF-IDF Matrix: (826, 101)
Shape of Metadata:      (826, 2)
Shape of Combined Data: (826, 103)


3. Multi-Task Learning

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

A. Prepare Data Splits

In [16]:
# Define our target variables
y_issue = df_clean['issue_type']
y_urgency = df_clean['urgency_level']

# Split the data into Training and Testing sets
# We use the same random_state to ensure X_test corresponds to both y_issue_test and y_urgency_test
X_train, X_test, y_issue_train, y_issue_test, y_urgency_train, y_urgency_test = train_test_split(
    X_combined, 
    y_issue, 
    y_urgency, 
    test_size=0.25, 
    random_state=42
)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape:  {X_test.shape}")

Training Data Shape: (619, 103)
Testing Data Shape:  (207, 103)


B. Model 1: Issue Type Classifier

In [17]:
print("\n--- Training Issue Type Classifier ---")
clf_issue = RandomForestClassifier(n_estimators=100, random_state=42)
clf_issue.fit(X_train, y_issue_train)

# Predictions
y_issue_pred = clf_issue.predict(X_test)

# Evaluation
print("Issue Type Classification Report:")
print(classification_report(y_issue_test, y_issue_pred))


--- Training Issue Type Classifier ---
Issue Type Classification Report:
                    precision    recall  f1-score   support

    Account Access       1.00      1.00      1.00        32
   Billing Problem       1.00      1.00      1.00        24
   General Inquiry       1.00      1.00      1.00        32
Installation Issue       1.00      1.00      1.00        37
     Late Delivery       1.00      1.00      1.00        20
    Product Defect       1.00      1.00      1.00        37
        Wrong Item       1.00      1.00      1.00        25

          accuracy                           1.00       207
         macro avg       1.00      1.00      1.00       207
      weighted avg       1.00      1.00      1.00       207



C. Model 2: Urgency Level Classifier

In [18]:
print("\n--- Training Urgency Level Classifier ---")
# Using class_weight='balanced' to handle potential imbalance (e.g., fewer 'Critical' tickets)
clf_urgency = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf_urgency.fit(X_train, y_urgency_train)

# Predictions
y_urgency_pred = clf_urgency.predict(X_test)

# Evaluation
print("Urgency Level Classification Report:")
print(classification_report(y_urgency_test, y_urgency_pred))


--- Training Urgency Level Classifier ---
Urgency Level Classification Report:
              precision    recall  f1-score   support

        High       0.35      0.36      0.36        77
         Low       0.27      0.30      0.28        61
      Medium       0.32      0.28      0.29        69

    accuracy                           0.31       207
   macro avg       0.31      0.31      0.31       207
weighted avg       0.31      0.31      0.31       207



4. Entity Extraction

A. Define Knowledge Bases

In [19]:
products = df_clean['product'].str.lower().unique().tolist()
complaint_keywords = ['broken', 'late', 'error', 'fail', 'crash', 'damaged', 'flickering', 'stuck', 'slow','malfunction','issue','problem','not working','stopped working','unresponsive','delay','missing','incorrect','unable','disconnect','overheated','repair','replace','refund','warranty','faulty','defective','glitch','hang','freeze','lag','unusable','corrupt','virus','spyware','adware','phishing','scam','spam','hijack','breach','intrusion','theft','loss','payment issue','instead','insufficient','unauthorized','billing','charge','invoice','subscription']

B. Define Extraction Function

In [20]:
def extract_entities(text):
    text_lower = str(text).lower()
    entities = {
        'products': [],
        'dates': [],
        'complaint_keywords': []
    }
    
    # --- A. Extract Products (Keyword Matching) ---
    # Check if any known product appears in the text
    for product in products:
        # Use regex boundary \b to avoid partial matches (e.g., preventing "net" matching inside "internet")
        if re.search(r'\b' + re.escape(product) + r'\b', text_lower):
            entities['products'].append(product)
            
    # --- B. Extract Complaint Keywords ---
    for keyword in complaint_keywords:
        if re.search(r'\b' + re.escape(keyword) + r'\b', text_lower):
            entities['complaint_keywords'].append(keyword)
            
    # --- C. Extract Dates (Regex Patterns) ---
    # Pattern 1: YYYY-MM-DD or DD-MM-YYYY or similar (e.g., 2023-10-05, 10/05/2023)
    date_pattern_num = r'\b\d{1,4}[-/]\d{1,2}[-/]\d{2,4}\b'
    
    # Pattern 2: Textual dates (e.g., "Jan 5th", "5 October")
    date_pattern_text = r'(?:\d{1,2}(?:st|nd|rd|th)?\s+)?(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2}(?:st|nd|rd|th)?(?:,\s+\d{4})?'
    
    # Find all matches
    matches_num = re.findall(date_pattern_num, text)
    matches_text = re.findall(date_pattern_text, text, re.IGNORECASE)
    
    entities['dates'] = matches_num + matches_text
    
    return entities

C. Apply to DataFrame

In [21]:
# We apply this to the original 'ticket_text' to preserve formatting/casing for date extraction if needed
df_clean['extracted_entities'] = df_clean['ticket_text'].apply(extract_entities)

D. Display Results

In [22]:
for index, row in df_clean.head().iterrows():
    print(f"Ticket: {row['ticket_text']}")
    print(f"Entities: {row['extracted_entities']}")
    print("-" * 30)

Ticket: Payment issue for my SmartWatch V2. I was underbilled for order #29224.
Entities: {'products': ['smartwatch v2'], 'dates': [], 'complaint_keywords': ['issue', 'payment issue']}
------------------------------
Ticket: I ordered SoundWave 300 but got EcoBreeze AC instead. My order number is #36824.
Entities: {'products': ['soundwave 300', 'ecobreeze ac'], 'dates': [], 'complaint_keywords': ['instead']}
------------------------------
Ticket: Facing installation issue with PhotoSnap Cam. Setup fails at step 1.
Entities: {'products': ['photosnap cam'], 'dates': [], 'complaint_keywords': ['issue']}
------------------------------
Ticket: Can you tell me more about the PhotoSnap Cam warranty? Also, is it available in red?
Entities: {'products': ['photosnap cam'], 'dates': [], 'complaint_keywords': ['warranty']}
------------------------------
Ticket:  is malfunction. It stopped working after just 7 days.
Entities: {'products': [], 'dates': [], 'complaint_keywords': ['malfunction', 'stopp

5. Integration

In [23]:
def process_ticket(raw_text):
    """
    End-to-end pipeline for a single support ticket.
    
    Args:
        raw_text (str): The raw customer support ticket text.
        
    Returns:
        dict: A dictionary containing predictions and extracted entities.
    """
    # -------------------------------------------------------
    # 1. Preprocessing
    # -------------------------------------------------------
    # Use the cleaning function defined in Step 1
    cleaned_text = clean_and_preprocess(raw_text)
    
    # -------------------------------------------------------
    # 2. Feature Engineering
    # -------------------------------------------------------
    # A. TF-IDF Vectorization
    # Note: We wrap cleaned_text in a list [] because transform expects an iterable
    tfidf_vec = tfidf_vectorizer.transform([cleaned_text])
    
    # B. Metadata Extraction
    ticket_len = len(raw_text)
    # Use the Sentiment Analyzer defined in Step 2
    sentiment = sia.polarity_scores(str(raw_text))['compound']
    
    # Create a 2D array for metadata [[len, sentiment]]
    meta_features = np.array([[ticket_len, sentiment]])
    
    # C. Combine Features
    # Stack the sparse TF-IDF vector with the dense metadata
    X_input = hstack([tfidf_vec, meta_features])
    
    # -------------------------------------------------------
    # 3. Model Prediction
    # -------------------------------------------------------
    # [0] is used to get the string value from the prediction array
    predicted_issue = clf_issue.predict(X_input)[0]
    predicted_urgency = clf_urgency.predict(X_input)[0]
    
    # -------------------------------------------------------
    # 4. Entity Extraction
    # -------------------------------------------------------
    # Use the extraction function defined in Step 4
    entities = extract_entities(raw_text)
    
    # -------------------------------------------------------
    # 5. Construct Output
    # -------------------------------------------------------
    result = {
        "raw_text": raw_text,
        "predicted_issue_type": predicted_issue,
        "predicted_urgency_level": predicted_urgency,
        "extracted_entities": entities
    }
    
    return result

In [24]:
# Test with a new, unseen ticket
new_ticket = "Vision LED TV is no response. It stopped working after just 9 days"


result = process_ticket(new_ticket)

# Pretty print the result
import json
print(json.dumps(result, indent=4))

{
    "raw_text": "Vision LED TV is no response. It stopped working after just 9 days",
    "predicted_issue_type": "Product Defect",
    "predicted_urgency_level": "Low",
    "extracted_entities": {
        "products": [
            "vision led tv"
        ],
        "dates": [],
        "complaint_keywords": [
            "stopped working"
        ]
    }
}


In [25]:
import gradio as gr

def gradio_wrapper(ticket_text):
    """
    Wrapper function to format the output for Gradio.
    """
    # 1. Get results from the main pipeline function
    result = process_ticket(ticket_text)
    
    # 2. Return the specific components
    # We return them in the order corresponding to the 'outputs' list below
    return (
        result['predicted_issue_type'], 
        result['predicted_urgency_level'], 
        result['extracted_entities']
    )

# -----------------------------------------------------------
# Define the Gradio Interface
# -----------------------------------------------------------
app = gr.Interface(
    fn=gradio_wrapper,
    inputs=gr.Textbox(
        lines=5, 
        placeholder="e.g., My internet is down and I need a refund...", 
        label="Customer Ticket Text"
    ),
    outputs=[
        gr.Label(label="Predicted Issue Type"),
        gr.Label(label="Predicted Urgency Level"),
        gr.JSON(label="Extracted Entities")
    ],
    title="Support Ticket AI Classifier",
    description="Enter a customer support message to automatically classify the issue, assess urgency, and extract key details (products, dates, etc.).",
    theme="default",
    examples=[
        ["My laptop screen is flickering constantly!!"],
        ["I need a refund for order #998877. It arrived broken."],
        ["URGENT: Server crash in the main database."]
    ]
)

# -----------------------------------------------------------
# Launch the App
# -----------------------------------------------------------
if __name__ == "__main__":
    # set share=True to create a public link (accessible from other devices)
    app.launch(share=False)

C:\Users\ramku\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ramku\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
